# LoRA the hard way

Gemma 3 1B learns the **Titan register** from 50 examples. Input: `thanos: <statement>`.

1. LoRA by hand, in plain PyTorch
2. The same adapter in PEFT, checked exactly
3. Every `LoraConfig` parameter: what it changes in our code, and a short run
4. Save, load, merge
5. Titan from both LoRAs

## Token

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "set HF_TOKEN"

## Setup

In [ ]:
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q "transformers==5.17.0" "peft==0.21.2"

In [ ]:
import gc, json, math, random, re, shutil, statistics, time, urllib.request
from collections import Counter

import torch
import torch.nn as nn
import transformers, peft
from huggingface_hub import HfApi, hf_hub_download, login
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "google/gemma-3-1b-it"
DATA = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/lora-the-hard-way/data/titan_v1/"
TAG, SEED = "thanos", 42
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DEVICE = torch.device("cuda:0")

login(HF_TOKEN)
api = HfApi(token=HF_TOKEN)
print(transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))

## Checkpoint mirror (private Hub repo)

In [ ]:
HUB_REPO, RUN_NAME = "skamalj/titan-stage1", "lora-the-hard-way-v1"


def hub_path(p):
    return f"runs/{RUN_NAME}/{os.path.relpath(p, WORK)}"


def hub_put(p):
    try:
        if os.path.isdir(p):
            api.upload_folder(repo_id=HUB_REPO, folder_path=p, path_in_repo=hub_path(p))
        else:
            api.upload_file(repo_id=HUB_REPO, path_or_fileobj=p, path_in_repo=hub_path(p))
    except Exception as e:
        print("  (mirror failed:", type(e).__name__, ")")


def hub_get(p):
    if os.path.exists(p):
        return True
    try:
        files = [f for f in api.list_repo_files(HUB_REPO) if f == hub_path(p) or f.startswith(hub_path(p) + "/")]
        for f in files:
            dst = p if f == hub_path(p) else os.path.join(p, os.path.relpath(f, hub_path(p)))
            os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
            shutil.copy(hf_hub_download(HUB_REPO, f, token=HF_TOKEN), dst)
        return bool(files)
    except Exception as e:
        print("  (restore failed:", type(e).__name__, ")")
        return False

## Data

- **train:** 50 pairs, 5 per rhetorical move, seed 42;
- **validation:** 100 pairs;
- **test:** 100 pairs.

In [ ]:
def fetch(split):
    rows = [json.loads(l) for l in urllib.request.urlopen(DATA + f"{split}.jsonl").read().decode().splitlines() if l]
    return [{"id": r["id"], "source": r["source"], "target": r["target"], "move": r["move"]} for r in rows]


rng = random.Random(SEED)
by_move = {}
for r in fetch("train"):
    by_move.setdefault(r["move"], []).append(r)
TRAIN = sorted([r for m in sorted(by_move) for r in rng.sample(by_move[m], 5)], key=lambda r: r["id"])
VAL, TEST = fetch("validation"), fetch("test")
print(len(TRAIN), len(VAL), len(TEST))
print(f"{TAG}: {TRAIN[0]['source']}\n  -> {TRAIN[0]['target']}")

# Part 1: LoRA by hand

## The frozen base model

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)


def load_base(dtype=torch.bfloat16):
    m = AutoModelForCausalLM.from_pretrained(MODEL, token=HF_TOKEN, dtype=dtype, device_map={"": 0},
                                             attn_implementation="eager")
    for p in m.parameters():
        p.requires_grad_(False)
    return m


model = load_base()
TOTAL = sum(p.numel() for p in model.parameters())
print(f"{TOTAL / 1e6:.0f} M parameters, all frozen")

## Prompt, loss, generation (for any model)

In [ ]:
EOS = tok.convert_tokens_to_ids("<end_of_turn>")


def prompt_ids(row):
    text = tok.apply_chat_template([{"role": "user", "content": f"{TAG}: {row['source']}"}],
                                   tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False)["input_ids"]


def batch(rows):
    seqs, labels = [], []
    for r in rows:
        p = prompt_ids(r)
        t = tok(r["target"], add_special_tokens=False)["input_ids"][:128] + [EOS]
        seqs.append(p + t)
        labels.append([-100] * len(p) + t)
    T = max(map(len, seqs))
    ids = torch.tensor([s + [tok.pad_token_id] * (T - len(s)) for s in seqs], device=DEVICE)
    mask = torch.tensor([[1] * len(s) + [0] * (T - len(s)) for s in seqs], device=DEVICE)
    lab = torch.tensor([l + [-100] * (T - len(l)) for l in labels], device=DEVICE)
    return ids, mask, lab


def loss_on(m, rows):
    ids, mask, lab = batch(rows)
    return m(input_ids=ids, attention_mask=mask, labels=lab).loss


@torch.no_grad()
def val_loss(m, rows=None, bs=8):
    rows = rows or VAL
    m.eval()
    return round(sum(loss_on(m, rows[i:i + bs]).item() * len(rows[i:i + bs]) for i in range(0, len(rows), bs)) / len(rows), 4)


@torch.no_grad()
def translate(m, rows, bs=16):
    m.eval()
    outs = []
    for i in range(0, len(rows), bs):
        seqs = [prompt_ids(r) for r in rows[i:i + bs]]
        T = max(map(len, seqs))
        ids = torch.tensor([[tok.pad_token_id] * (T - len(s)) + s for s in seqs], device=DEVICE)
        mask = torch.tensor([[0] * (T - len(s)) + [1] * len(s) for s in seqs], device=DEVICE)
        gen = m.generate(input_ids=ids, attention_mask=mask, do_sample=False, max_new_tokens=128,
                         eos_token_id=[EOS, tok.eos_token_id], pad_token_id=tok.pad_token_id)
        outs += [tok.decode(g, skip_special_tokens=True).strip() for g in gen[:, T:].tolist()]
    return outs

## Base logits on one sentence (the before picture)

In [ ]:
one_ids, one_mask, _ = batch(TEST[:1])
with torch.no_grad():
    model.eval()
    BASE_LOGITS = model(input_ids=one_ids, attention_mask=one_mask).logits.float().cpu()

## The LoRA layer

`y = W x + (α / r) · B A dropout(x)`
- **W:** frozen.
- **A** (r × in): random at the start.
- **B** (out × r): zero at the start.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.r, self.scaling = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y = self.base(x)
        delta = (self.drop(x).float() @ self.A.T) @ self.B.T
        return y + (self.scaling * delta).to(y.dtype)

## Settings

In [ ]:
R, ALPHA, DROPOUT = 16, 32, 0.15
LR, WEIGHT_DECAY = 1e-4, 0.05
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
EPOCHS, PATIENCE, MICRO, ACCUM = 15, 2, 4, 2

## Inject into every attention and MLP projection

In [ ]:
def inject(m, r=R, alpha=ALPHA, dropout=DROPOUT, targets=TARGETS):
    torch.manual_seed(SEED)
    for module in list(m.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, LoRALinear(child, r, alpha, dropout))
    return {n: x for n, x in m.named_modules() if isinstance(x, LoRALinear)}


LORA = inject(model)
TRAINABLE = [p for x in LORA.values() for p in (x.A, x.B)]
n_train = sum(p.numel() for p in TRAINABLE)
print(f"{len(LORA)} layers wrapped | trainable {n_train:,} = {100 * n_train / TOTAL:.2f} % of the model")

## Check: with B = 0 nothing changes

In [ ]:
with torch.no_grad():
    model.eval()
    after = model(input_ids=one_ids, attention_mask=one_mask).logits.float().cpu()
print("max |logit difference| =", (after - BASE_LOGITS).abs().max().item())

## Training loop (shared by every run in this notebook)

In [ ]:
def train(m, params, lr=LR, wd=WEIGHT_DECAY, epochs=EPOCHS):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    steps = epochs * math.ceil(len(TRAIN) / (MICRO * ACCUM))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))
    log = [{"epoch": 0, "validation_loss": val_loss(m)}]
    best, since = (log[0]["validation_loss"], 0, [p.detach().cpu().clone() for p in params]), 0
    for epoch in range(1, epochs + 1):
        m.train()
        order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
        losses = []
        for i, start in enumerate(range(0, len(order), MICRO), 1):
            loss = loss_on(m, order[start:start + MICRO])
            (loss / ACCUM).backward()
            losses.append(loss.item())
            if i % ACCUM == 0 or start + MICRO >= len(order):
                opt.step(); sched.step(); opt.zero_grad()
        log.append({"epoch": epoch, "train_loss": round(statistics.mean(losses), 4), "validation_loss": val_loss(m)})
        if log[-1]["validation_loss"] < best[0]:
            best, since = (log[-1]["validation_loss"], epoch, [p.detach().cpu().clone() for p in params]), 0
        else:
            since += 1
            if since >= PATIENCE:
                break
    for p, s in zip(params, best[2]):
        p.data.copy_(s.to(p.device))
    return log, best[1], best[0]

## Train our hand-built LoRA

In [ ]:
HAND_FILE = f"{WORK}/hand_lora.pt"
if hub_get(HAND_FILE):
    saved = torch.load(HAND_FILE)
    for p, s in zip(TRAINABLE, saved["state"]):
        p.data.copy_(s.to(p.device))
    HAND_LOG = saved["log"]
else:
    t0 = time.perf_counter()
    HAND_LOG, ep, best = train(model, TRAINABLE)
    torch.save({"state": [p.detach().cpu() for p in TRAINABLE], "log": HAND_LOG}, HAND_FILE)
    hub_put(HAND_FILE)
    print(f"kept epoch {ep} (validation loss {best}) | {time.perf_counter() - t0:.0f}s")
for e in HAND_LOG:
    print(e)

## Test translations and facts

In [ ]:
def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}




def facts(rows, outs):
    sel = [fact_checks(r["source"], o) for r, o in zip(rows, outs)]
    return {k: round(statistics.mean(x[k] for x in sel if x[k] is not None), 3)
            for k in ("numbers_ok", "dates_ok", "names_ok") if any(x[k] is not None for x in sel)}

In [ ]:
TEST_OUT = translate(model, TEST)
print("facts kept:", facts(TEST, TEST_OUT))
for r, o in list(zip(TEST, TEST_OUT))[:4]:
    print(f"\n{TAG}: {r['source']}\n  -> {o}")

# Part 2: the same adapter in PEFT

Both models are compared in float32, made the same way (bf16 load, then `.float()`). Some buffers are computed at load time in the load dtype: Gemma's embedding scale √1152 = 33.94 becomes 34.0 in bf16.

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model

HAND_STATE = [p.detach().cpu().clone() for p in TRAINABLE]       # kept for the closing demo
model = model.float()
pmodel = get_peft_model(load_base().float(),
                        LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, target_modules=list(TARGETS),
                                   task_type="CAUSAL_LM"))
pmodel.print_trainable_parameters()

## Copy our A and B into PEFT's lora_A and lora_B

In [ ]:
for name, x in LORA.items():
    t = pmodel.base_model.model.get_submodule(name)
    t.lora_A["default"].weight.data.copy_(x.A.data)
    t.lora_B["default"].weight.data.copy_(x.B.data)

## Check 1: one layer, the same random input

In [ ]:
name = next(iter(LORA))
x = torch.randn(4, LORA[name].base.in_features, device=DEVICE)
with torch.no_grad():
    model.eval(); pmodel.eval()
    print(name, "| max |difference| =", (LORA[name](x) - pmodel.base_model.model.get_submodule(name)(x)).abs().max().item())

## Check 2: the whole model, one sentence

In [ ]:
with torch.no_grad():
    ours = model(input_ids=one_ids, attention_mask=one_mask).logits
    theirs = pmodel(input_ids=one_ids, attention_mask=one_mask).logits
print("max |logit difference| =", (ours - theirs).abs().max().item())

## Check 3: translations

In [ ]:
a, b = translate(model, TEST[:10]), translate(pmodel, TEST[:10])
print("same translations:", sum(x == y for x, y in zip(a, b)), "of 10")

## Merge by hand: `W' = W + (α / r) · B A`, against PEFT's `merge_and_unload()`

In [ ]:
with torch.no_grad():
    for name, x in LORA.items():
        x.base.weight.data += x.scaling * (x.B @ x.A)
        parent, attr = name.rsplit(".", 1)
        setattr(model.get_submodule(parent), attr, x.base)
    merged = pmodel.merge_and_unload()
    diff = max((model.get_submodule(n).weight - merged.get_submodule(n).weight).abs().max().item() for n in LORA)
print("max |merged weight difference| over all", len(LORA), "layers =", diff)
del model, pmodel, merged
gc.collect(); torch.cuda.empty_cache()

# Part 3: every `LoraConfig` parameter

Each run trains a fresh PEFT adapter on the same 50 pairs, with early stopping, and records the trainable parameters, the best validation loss and three test translations.

In [ ]:
RUNS_FILE = f"{WORK}/runs.json"
RUNS = json.load(open(RUNS_FILE)) if hub_get(RUNS_FILE) else {}
BASE_KW = dict(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, target_modules=list(TARGETS), task_type="CAUSAL_LM")


def run(name, keep=False, **changes):
    """Train one PEFT config (BASE_KW with `changes`); keep=True returns the model instead of freeing it."""
    if name in RUNS and not keep:
        print(name, "(done earlier)", {k: RUNS[name][k] for k in ("trainable", "best_validation_loss", "best_epoch")})
        return None
    pm = get_peft_model(load_base(), LoraConfig(**{**BASE_KW, **changes}))
    params = [p for p in pm.parameters() if p.requires_grad]
    t0 = time.perf_counter()
    log, ep, best = train(pm, params)
    RUNS[name] = {"changes": {k: str(v) for k, v in changes.items()}, "trainable": sum(p.numel() for p in params),
                  "best_validation_loss": best, "best_epoch": ep, "seconds": round(time.perf_counter() - t0),
                  "samples": translate(pm, TEST[:3])}
    json.dump(RUNS, open(RUNS_FILE, "w"), ensure_ascii=False)
    hub_put(RUNS_FILE)
    print(name, {k: RUNS[name][k] for k in ("trainable", "best_validation_loss", "best_epoch", "seconds")})
    if keep:
        return pm
    del pm
    gc.collect(); torch.cuda.empty_cache()

## The baseline: our recipe, trained by PEFT (kept for Parts 4 and 5)

In [ ]:
PEFT_DIR = f"{WORK}/peft_titan"
if hub_get(PEFT_DIR) and "baseline" in RUNS:
    print("baseline adapter restored from the mirror")
else:
    pm = run("baseline", keep=True)
    pm.save_pretrained(PEFT_DIR)
    hub_put(PEFT_DIR)
    del pm
    gc.collect(); torch.cuda.empty_cache()

## `r`

The inner size of A and B; trainable parameters grow linearly with it.

- **In our code:** `self.A`: r × in, `self.B`: out × r
- α = 2r keeps the scale α/r fixed, so only the size changes.

In [ ]:
run("r4", r=4, lora_alpha=8)
run("r64", r=64, lora_alpha=128)

## `lora_alpha`

Scales the update: `scaling = alpha / r`.

- **In our code:** `self.scaling = alpha / r`
- At r = 16: scale 1 and 4 (the baseline is 2).

In [ ]:
run("alpha16", lora_alpha=16)
run("alpha64", lora_alpha=64)

## `use_rslora`

Rank-stabilised scaling: `alpha / sqrt(r)` instead of `alpha / r`.

- **In our code:** `self.scaling = alpha / math.sqrt(r)`
- At r = 64: scale 4 instead of 0.5.

In [ ]:
run("rslora_r64", r=64, lora_alpha=32, use_rslora=True)

## `lora_dropout`

Dropout on the input to A only; W sees the clean input.

- **In our code:** `self.drop`

In [ ]:
run("dropout0", lora_dropout=0.0)

## `target_modules`

Which layers get an adapter (names, a list, or a regex); `exclude_modules` removes some.

- **In our code:** `TARGETS`

In [ ]:
run("attention_only", target_modules=["q_proj","k_proj","v_proj","o_proj"])
run("mlp_only", target_modules=["gate_proj","up_proj","down_proj"])

## `layers_to_transform`

Adapters only in these layer indices; `layers_pattern` names the layer list (`layers`).

- **In our code:** `if layer_index in keep` in `inject`
- Gemma 3 1B has 26 layers.

In [ ]:
run("last8", layers_to_transform=list(range(18, 26)))

## `rank_pattern / alpha_pattern`

Per-module r and α, overriding the defaults by regex.

- **In our code:** a different `r` per target

In [ ]:
run("mlp_r32_attn_r8", r=8, lora_alpha=16, rank_pattern={"gate_proj": 32, "up_proj": 32, "down_proj": 32}, alpha_pattern={"gate_proj": 64, "up_proj": 64, "down_proj": 64})

## `init_lora_weights`

How A and B start. `True`: Kaiming A, zero B (starts as the base). `gaussian`: normal A, zero B. `pissa`: A and B from the top singular vectors of W, which are removed from W. `olora`: from a QR of W. `loftq`: for quantised bases. `eva`, `corda`: data-driven, from activations.

- **In our code:** `nn.init.kaiming_uniform_(self.A, ...)` and `torch.zeros` for B

In [ ]:
run("gaussian", init_lora_weights="gaussian")
run("pissa", init_lora_weights="pissa_niter_4")

## `use_dora`

DoRA: the weight is split into a magnitude (one number per output) and a direction; LoRA updates the direction, and the magnitude is trained too.

- **In our code:** `y = m · (W + s·B A) x / ||W + s·B A||` per output row

In [ ]:
run("dora", use_dora=True)

## PiSSA by hand (what `init_lora_weights="pissa"` does)

Take the top-r singular vectors of W as A and B, and subtract them from W, so W + B A is unchanged at the start.

In [ ]:
W = load_base().model.layers[0].self_attn.q_proj.weight.float()
U, S, Vh = torch.linalg.svd(W, full_matrices=False)
B_init, A_init = U[:, :R] * S[:R].sqrt(), S[:R].sqrt()[:, None] * Vh[:R]
W_res = W - B_init @ A_init
print("W + B A unchanged:", (W_res + B_init @ A_init - W).abs().max().item(),
      "| share of W's energy in the top", R, "directions:", round((S[:R] ** 2).sum().item() / (S ** 2).sum().item(), 3))
del W, U, S, Vh, W_res
gc.collect(); torch.cuda.empty_cache()

## The rest of `LoraConfig`

| Parameter | What it does |
|---|---|
| `bias` | `"none"` / `"all"` / `"lora_only"`: also train the bias of the wrapped layers (Gemma's projections have none) |
| `lora_bias` | add a trainable bias to B |
| `modules_to_save` | train full copies of extra modules, e.g. a classification head |
| `trainable_token_indices` | train only some rows of the embedding, e.g. one new token |
| `task_type` | which PEFT wrapper is used (`CAUSAL_LM` here) |
| `inference_mode` | freeze the adapter on load |
| `fan_in_fan_out` | for layers that store W transposed (GPT-2's `Conv1D`) |
| `revision`, `megatron_*` | the base model revision; Megatron-parallel layers |

In [ ]:
pm = get_peft_model(load_base(), LoraConfig(**{**BASE_KW, "trainable_token_indices": {"embed_tokens": [tok.convert_tokens_to_ids("<unused0>")]}}))
pm.print_trainable_parameters()
del pm
gc.collect(); torch.cuda.empty_cache()

# Part 4: save, load, merge

## Files on disk

In [ ]:
print(sorted(os.listdir(PEFT_DIR)))
print(json.dumps({k: v for k, v in json.load(open(f"{PEFT_DIR}/adapter_config.json")).items() if v not in (None, False, {}, [])},
                 indent=1))

## Load onto a fresh base, then merge

In [ ]:
loaded = PeftModel.from_pretrained(load_base(), PEFT_DIR)
before = translate(loaded, TEST[:3])
merged = loaded.merge_and_unload()
after = translate(merged, TEST[:3])
print("same translations after merging:", sum(x == y for x, y in zip(before, after)), "of 3")

# Part 5: Titan, from both LoRAs

In [ ]:
hand = load_base()
for p, s in zip([q for x in inject(hand).values() for q in (x.A, x.B)], HAND_STATE):
    p.data.copy_(s.to(p.device))
by_hand, by_peft = translate(hand, TEST[:3]), translate(merged, TEST[:3])
for r, a, b in zip(TEST[:3], by_hand, by_peft):
    print(f"{TAG}: {r['source']}\n  by hand: {a}\n  PEFT   : {b}\n")

## All runs

In [ ]:
print(f"{'run':18s} {'trainable':>12s} {'best val loss':>14s} {'epoch':>6s}")
for k, v in RUNS.items():
    print(f"{k:18s} {v['trainable']:>12,} {v['best_validation_loss']:>14.4f} {v['best_epoch']:>6d}")